# Homework: Quantum Repeater Lab — Teleportation, Swapping, and Long-Distance Entanglement
**Lecture 4.4 — Quantum Teleportation and Superdense Coding**

---

**Briefing**

You are an engineer at a quantum networking startup. Your job: build a quantum repeater prototype that can distribute entanglement across a chain of nodes. The building blocks are (1) quantum teleportation and (2) entanglement swapping. In this lab you'll build both, verify they work, and then push them to their limits with noise.

---

**Instructions**

- Complete every cell marked `### YOUR CODE HERE ###`.
- Do **not** rename variables that appear in the prompts — those names may be checked by the grader.
- Short-answer cells are graded on correctness and clarity, not length.
- Before submitting: `Kernel → Restart & Run All` and confirm zero errors.
- Upload this `.ipynb` file to Gradescope.

---

| Part | Topic | Points |
|------|-------|--------|
| A | Building and verifying teleportation | 20 |
| B | Teleportation with a different Bell pair | 20 |
| C | Entanglement swapping | 25 |
| D | Noisy quantum repeater chain | 15 |
| **Total** | | **80** |

In [ ]:
# Run this cell first — standard imports
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator, partial_trace, state_fidelity
from qiskit_aer import AerSimulator
import numpy as np
import matplotlib.pyplot as plt

simulator = AerSimulator()

---
## Part A — Building and Verifying Teleportation (20 pts)

### Background

In the teleportation protocol, Alice wants to send an unknown qubit $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ to Bob. They share a Bell pair $|\Phi^+\rangle$. The circuit has three qubits:

- Qubit 0: Alice's unknown state $|\psi\rangle$
- Qubit 1: Alice's half of $|\Phi^+\rangle$
- Qubit 2: Bob's half of $|\Phi^+\rangle$

The protocol:
1. Alice applies CNOT (qubit 0 → qubit 1), then H on qubit 0.
2. Alice measures qubits 0 and 1, getting bits $(m_0, m_1)$.
3. Bob applies corrections: $X^{m_1} Z^{m_0}$ to qubit 2.

We'll use the **deferred measurement** version: instead of measuring and classically conditioning, we use controlled-X and controlled-Z gates from Alice's qubits to Bob's qubit, then measure everything at the end. The two approaches are equivalent by the deferred measurement principle.

### A1 (8 pts) — Build the teleportation circuit

Write a function `teleportation_circuit(input_gate)` that returns a 3-qubit `QuantumCircuit` implementing teleportation with deferred measurement.

The argument `input_gate` is a `QuantumCircuit` on 1 qubit that prepares the state Alice wants to teleport (applied to qubit 0).

Your circuit should:
1. Apply `input_gate` to qubit 0 to prepare $|\psi\rangle$.
2. Create $|\Phi^+\rangle$ between qubits 1 and 2 (H on qubit 1, CNOT 1→2).
3. Apply Alice's operations: CNOT (0→1), then H on qubit 0.
4. Apply Bob's corrections via deferred measurement: CX (1→2), CZ (0→2).

**Do not** add measurements — we'll inspect the statevector directly.

Draw the circuit for a test input where `input_gate` applies $R_y(\pi/4)$ to qubit 0.

In [ ]:
def teleportation_circuit(input_gate):
    """
    Build a 3-qubit teleportation circuit with deferred measurement.
    
    Args:
        input_gate: QuantumCircuit on 1 qubit that prepares |psi>
    
    Returns:
        QuantumCircuit on 3 qubits
    """
    qc = QuantumCircuit(3)
    
    # Step 1: Prepare Alice's input state on qubit 0
    qc.append(input_gate, [0])
    qc.barrier(label="input")
    
    ### YOUR CODE HERE ###
    
    # Step 2: Create Bell pair |Phi+> between qubits 1 and 2
    # ...
    
    qc.barrier(label="Bell pair")
    
    # Step 3: Alice's operations (CNOT 0→1, then H on qubit 0)
    # ...
    
    qc.barrier(label="Alice")
    
    # Step 4: Bob's corrections (CX 1→2, CZ 0→2)
    # ...
    
    return qc


# Test: prepare |psi> = Ry(pi/4)|0>
prep = QuantumCircuit(1)
prep.ry(np.pi/4, 0)

qc_test = teleportation_circuit(prep)
qc_test.draw('mpl')

### A2 (8 pts) — Verify teleportation for multiple input states

Verify your circuit using the **$U^\dagger$ trick**: if teleportation works, applying $U^\dagger$ to Bob's qubit after teleportation should return it to $|0\rangle$.

For each of these four input states, build the teleportation circuit, append $U^\dagger$ to qubit 2, and check that qubit 2 ends up in $|0\rangle$.

| State | Preparation gate $U$ |
|-------|---------------------|
| $\lvert 0\rangle$ | $I$ (identity) |
| $\lvert 1\rangle$ | $X$ |
| $\lvert +\rangle$ | $H$ |
| $\cos(\pi/8)\lvert 0\rangle + \sin(\pi/8)\lvert 1\rangle$ | $R_y(\pi/4)$ |

For each state:
1. Build teleportation circuit with `input_gate` = $U$.
2. Append $U^\dagger$ to qubit 2 (use `input_gate.inverse()`).
3. Compute the statevector.
4. Extract Bob's qubit (qubit 2) using `partial_trace` over qubits 0 and 1.
5. Check that Bob's qubit has probability 1 of being in $|0\rangle$.

Store the four probabilities of $|0\rangle$ for Bob's qubit in a list `p0_values`. All should be 1.0 (to numerical precision).

In [ ]:
### YOUR CODE HERE ###

test_states = {
    "|0>": QuantumCircuit(1),         # identity
    "|1>": None,                       # X gate
    "|+>": None,                       # H gate
    "Ry(pi/4)|0>": None,              # Ry(pi/4)
}

# Fill in the preparation gates
# ...

p0_values = []

for label, prep_gate in test_states.items():
    # Build teleportation circuit
    qc = teleportation_circuit(prep_gate)
    
    # Append U† to qubit 2
    qc.append(prep_gate.inverse(), [2])
    
    # Get statevector and trace out qubits 0 and 1
    sv = Statevector.from_instruction(qc)
    # partial_trace: trace out qubits [0, 1] to keep qubit 2
    rho_bob = partial_trace(sv, [0, 1])
    
    # Probability of |0> for Bob's qubit
    p0 = rho_bob.probabilities()[0]
    p0_values.append(p0)
    
    print(f"Input: {label:<15}  P(Bob = |0>) = {p0:.6f}  {'✓' if np.isclose(p0, 1.0) else '✗'}")

print(f"\nAll teleportations successful: {all(np.isclose(p, 1.0) for p in p0_values)}")

### A3 (4 pts) — Short answer

1. In the deferred measurement version, we replaced Alice's measurements + classical communication with CX and CZ gates. Why does this give the same result? (2–3 sentences, referencing the deferred measurement principle.)

2. The original qubit $|\psi\rangle$ is destroyed during teleportation. Where in the circuit does this happen, and why is this consistent with the no-cloning theorem? (2–3 sentences.)

**Your answer:**

*(double-click to edit)*

---
## Part B — Teleportation with a Different Bell Pair (20 pts)

### Background

In the lecture we used $|\Phi^+\rangle$ as the shared Bell pair. But in real experiments, the source might produce a different Bell state. For instance, parametric down-conversion in optics naturally produces $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$.

If Alice and Bob share $|\Psi^-\rangle$ instead of $|\Phi^+\rangle$, the protocol still works — but the **correction table changes**.

### B1 (8 pts) — Derive the new correction table

Work through the algebra (or use Qiskit) to find the correction table when the shared state is $|\Psi^-\rangle$ instead of $|\Phi^+\rangle$.

Write a function `teleportation_psi_minus(input_gate)` that implements teleportation using $|\Psi^-\rangle$ as the shared Bell pair, with the corrected Bob operations.

To prepare $|\Psi^-\rangle$ between qubits 1 and 2: apply X to qubit 1, then H to qubit 1, then CNOT(1→2).

**Hint:** The relationship between $|\Phi^+\rangle$ and $|\Psi^-\rangle$ is $|\Psi^-\rangle = (X \otimes I) \cdot (Z \otimes I) |\Phi^+\rangle$. So Bob needs to undo the extra $XZ$ that Alice's half picked up. The corrected gates are: CX(1→2) to handle the X bit, CZ(0→2) for the phase, plus an unconditional $X$ and $Z$ on qubit 2.

In [ ]:
def teleportation_psi_minus(input_gate):
    """
    Teleportation circuit using |Psi-> as the shared Bell pair.
    Uses deferred measurement.
    """
    qc = QuantumCircuit(3)
    
    # Prepare input on qubit 0
    qc.append(input_gate, [0])
    qc.barrier(label="input")
    
    ### YOUR CODE HERE ###
    
    # Prepare |Psi-> between qubits 1 and 2
    # (X on qubit 1, H on qubit 1, CNOT 1→2)
    # ...
    
    qc.barrier(label="Bell pair")
    
    # Alice's operations: same as before (CNOT 0→1, H on qubit 0)
    # ...
    
    qc.barrier(label="Alice")
    
    # Bob's corrections — you need to figure out the right gates!
    # Hint: start with the same CX(1→2) and CZ(0→2) as before,
    # then add unconditional corrections to account for |Psi-> vs |Phi+>
    # ...
    
    return qc


# Draw the circuit
prep_test = QuantumCircuit(1)
prep_test.ry(np.pi/4, 0)
teleportation_psi_minus(prep_test).draw('mpl')

### B2 (8 pts) — Verify the new protocol

Verify `teleportation_psi_minus` works for the same four test states as Part A2. Use the $U^\dagger$ trick.

Store the four probabilities of $|0\rangle$ for Bob's qubit in `p0_psi_minus`. All should be 1.0.

In [ ]:
### YOUR CODE HERE ###

p0_psi_minus = []

for label, prep_gate in test_states.items():
    qc = teleportation_psi_minus(prep_gate)
    qc.append(prep_gate.inverse(), [2])
    
    sv = Statevector.from_instruction(qc)
    rho_bob = partial_trace(sv, [0, 1])
    p0 = rho_bob.probabilities()[0]
    p0_psi_minus.append(p0)
    
    print(f"Input: {label:<15}  P(Bob = |0>) = {p0:.6f}  {'✓' if np.isclose(p0, 1.0) else '✗'}")

print(f"\nAll teleportations with |Ψ⁻⟩ successful: {all(np.isclose(p, 1.0) for p in p0_psi_minus)}")

### B3 (4 pts) — Short answer

Fill in the correction table for teleportation with $|\Psi^-\rangle$. Compare with the $|\Phi^+\rangle$ table from lecture.

| Alice measures (deferred) | Bob applies ($\lvert\Phi^+\rangle$) | Bob applies ($\lvert\Psi^-\rangle$) |
|:---:|:---:|:---:|
| $(0, 0)$ | $I$ | ? |
| $(0, 1)$ | $X$ | ? |
| $(1, 0)$ | $Z$ | ? |
| $(1, 1)$ | $XZ$ | ? |

In 2–3 sentences, explain the pattern: how does the correction table change when you switch from $|\Phi^+\rangle$ to $|\Psi^-\rangle$?

**Your answer:**

*(double-click to edit)*

---
## Part C — Entanglement Swapping (25 pts)

### Background

**Entanglement swapping** is teleportation where the "unknown state" is itself half of an entangled pair. The result: two particles that have never interacted become entangled.

Setup:
- Alice and Charlie share $|\Phi^+\rangle_{AC}$ (qubits 0, 1).
- Charlie and Bob share $|\Phi^+\rangle_{CB}$ (qubits 2, 3).
- Charlie performs a Bell measurement on his two qubits (1 and 2).
- After Charlie's measurement (and corrections), Alice (qubit 0) and Bob (qubit 3) are entangled!

This is the core of **quantum repeaters**: chaining entanglement swaps to distribute entanglement over long distances.

### C1 (10 pts) — Build the entanglement swapping circuit

Build a 4-qubit circuit that performs entanglement swapping using deferred measurement:

1. Create $|\Phi^+\rangle$ between qubits 0 and 1 (Alice–Charlie pair 1).
2. Create $|\Phi^+\rangle$ between qubits 2 and 3 (Charlie–Bob pair 2).
3. Charlie's Bell measurement on qubits 1 and 2: CNOT(1→2), then H on qubit 1.
4. Corrections on Bob's qubit 3: CX(2→3), CZ(1→3).

Store the circuit in `qc_swap`. Draw it.

In [ ]:
### YOUR CODE HERE ###

qc_swap = QuantumCircuit(4)

# Step 1: Bell pair between qubits 0 and 1
# ...

# Step 2: Bell pair between qubits 2 and 3
# ...

qc_swap.barrier(label="two Bell pairs")

# Step 3: Charlie's Bell measurement on qubits 1, 2
# ...

qc_swap.barrier(label="Charlie measures")

# Step 4: Corrections on qubit 3
# ...

qc_swap.draw('mpl')

### C2 (10 pts) — Verify Alice and Bob are entangled

After entanglement swapping, qubits 0 (Alice) and 3 (Bob) should be in the Bell state $|\Phi^+\rangle$.

Verify this:
1. Compute the statevector of `qc_swap`.
2. Trace out Charlie's qubits (1 and 2) to get the reduced density matrix of Alice and Bob (qubits 0 and 3).
3. Compute the **fidelity** of this reduced state with $|\Phi^+\rangle$ using `state_fidelity`.

Store the fidelity in `swap_fidelity`. It should be 1.0.

*Hint:* To create the target Bell state for fidelity comparison, use `Statevector.from_label('00')` evolved through a Bell circuit, or construct it directly as a numpy array.

In [ ]:
### YOUR CODE HERE ###

# Get statevector
sv_swap = Statevector.from_instruction(qc_swap)

# Trace out Charlie's qubits (1, 2) to get Alice-Bob state (qubits 0, 3)
rho_AB = partial_trace(sv_swap, [1, 2])

# Target: |Phi+> on 2 qubits
phi_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])

# Fidelity
swap_fidelity = state_fidelity(rho_AB, phi_plus)

print(f"Reduced density matrix of Alice (q0) and Bob (q3):")
print(np.round(rho_AB.data, 4))
print(f"\nFidelity with |Φ⁺⟩: {swap_fidelity:.6f}  {'✓' if np.isclose(swap_fidelity, 1.0) else '✗'}")
print(f"\nAlice and Bob are entangled — even though they never interacted!")

### C3 (5 pts) — Short answer

1. In entanglement swapping, Alice and Bob end up entangled even though their qubits never directly interacted. Does this allow faster-than-light communication? Why or why not? (2–3 sentences.)

2. Entanglement swapping is equivalent to "teleporting half of an entangled pair." Explain this interpretation: what is the "unknown state" being teleported, and who plays the role of sender/receiver? (2–3 sentences.)

**Your answer:**

*(double-click to edit)*

---
## Part D — Noisy Quantum Repeater Chain (15 pts)

### Background

In reality, the Bell pairs shared between neighboring nodes aren't perfect — noise degrades them. A simple noise model is the **depolarizing channel**, which replaces the state with the maximally mixed state with some probability:

$$\mathcal{E}(\rho) = (1-p)\rho + p \cdot \frac{I}{4}$$

where $p$ is the noise parameter and $I/4$ is the maximally mixed 2-qubit state.

When Bell pairs are noisy, each entanglement swap degrades the fidelity further. We'll build a repeater chain and study how fidelity drops with the number of nodes.

### D1 (10 pts) — Fidelity vs. chain length

Write a function `repeater_chain_fidelity(n_links, noise_p)` that:
1. Creates a chain of `n_links` Bell pairs (each between adjacent nodes).
2. Applies depolarizing noise to each Bell pair.
3. Performs `n_links - 1` entanglement swaps to connect the endpoints.
4. Returns the fidelity of the final end-to-end state with $|\Phi^+\rangle$.

Since building large quantum circuits gets unwieldy, we'll simulate this using **density matrices** directly.

The approach:
- Start with the density matrix of a noisy Bell pair: $\rho_{\text{noisy}} = (1-p)|\Phi^+\rangle\langle\Phi^+| + p \cdot I/4$.
- For a chain with $n$ links, the end-to-end fidelity after swapping is approximately $F_n \approx \left(\frac{1 + 3(1-p)^n}{4}\right)$ for small noise. But let's measure it numerically.

For a 2-link chain (the simplest case beyond a single link):
1. Create two noisy Bell pairs as a 4-qubit density matrix (tensor product).
2. Apply the entanglement swap operations (CNOT + H on the middle qubits, then corrections).
3. Trace out the middle qubits.
4. Compute fidelity of the remaining state with $|\Phi^+\rangle$.

For longer chains, repeat this process: take the end-to-end state from the previous step, tensor it with a new noisy Bell pair, swap, trace out, and continue.

Plot the fidelity vs. number of links for `noise_p` = 0.0, 0.05, 0.10, and 0.20. Use chains from 1 to 6 links.

Store your results in a dictionary `fidelity_data` with keys being noise values and values being lists of fidelities.

In [ ]:
from qiskit.quantum_info import DensityMatrix

def noisy_bell_pair(noise_p):
    """
    Create a noisy Bell pair density matrix.
    rho = (1-p)|Phi+><Phi+| + p * I/4
    """
    phi_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
    rho_pure = DensityMatrix(phi_plus)
    rho_mixed = DensityMatrix(np.eye(4) / 4)
    return DensityMatrix((1 - noise_p) * rho_pure.data + noise_p * rho_mixed.data)


def entanglement_swap_density(rho_4qubit):
    """
    Perform entanglement swap on a 4-qubit density matrix.
    Qubits: 0 (Alice), 1 (Charlie-left), 2 (Charlie-right), 3 (Bob)
    Charlie measures qubits 1, 2 (Bell measurement + corrections via deferred measurement).
    Returns the 2-qubit density matrix of qubits 0 and 3.
    """
    ### YOUR CODE HERE ###
    
    # Build the swap circuit on 4 qubits
    qc = QuantumCircuit(4)
    # Charlie's Bell measurement: CNOT(1→2), H on qubit 1
    # ...
    # Corrections: CX(2→3), CZ(1→3)
    # ...
    
    # Evolve the density matrix
    rho_after = rho_4qubit.evolve(qc)
    
    # Trace out Charlie's qubits (1, 2)
    rho_AB = partial_trace(rho_after, [1, 2])
    
    return rho_AB


def repeater_chain_fidelity(n_links, noise_p):
    """
    Compute end-to-end fidelity of an n-link repeater chain.
    """
    if n_links == 1:
        rho = noisy_bell_pair(noise_p)
        phi_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
        return state_fidelity(rho, phi_plus)
    
    ### YOUR CODE HERE ###
    
    # Start with the first link
    rho_chain = noisy_bell_pair(noise_p)
    
    for _ in range(n_links - 1):
        # Tensor with a new noisy Bell pair
        rho_new_link = noisy_bell_pair(noise_p)
        rho_4qubit = DensityMatrix(np.kron(rho_chain.data, rho_new_link.data))
        
        # Swap and trace out
        rho_chain = entanglement_swap_density(rho_4qubit)
    
    phi_plus = Statevector([1/np.sqrt(2), 0, 0, 1/np.sqrt(2)])
    return state_fidelity(rho_chain, phi_plus)


# Compute and plot
noise_values = [0.0, 0.05, 0.10, 0.20]
n_links_range = range(1, 7)
fidelity_data = {}

fig, ax = plt.subplots(figsize=(8, 5))

for noise_p in noise_values:
    fidelities = [repeater_chain_fidelity(n, noise_p) for n in n_links_range]
    fidelity_data[noise_p] = fidelities
    ax.plot(list(n_links_range), fidelities, 'o-', label=f'p = {noise_p}')

ax.set_xlabel('Number of links')
ax.set_ylabel('End-to-end fidelity with |Φ⁺⟩')
ax.set_title('Quantum Repeater: Fidelity vs. Chain Length')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Classical limit (F=0.5)')
ax.legend()
ax.set_ylim(0.3, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print table
print(f"{'Links':<8}", end='')
for p in noise_values:
    print(f"p={p:<8.2f}", end='')
print()
for i, n in enumerate(n_links_range):
    print(f"{n:<8}", end='')
    for p in noise_values:
        print(f"{fidelity_data[p][i]:<10.4f}", end='')
    print()

### D2 (5 pts) — Short answer

1. For the noisy cases, the fidelity drops with each additional link. Once it falls below $F = 0.5$, the entanglement is no better than a classical correlation (the state is separable). For $p = 0.20$, roughly how many links can you chain before hitting this limit? (Read from your plot.)

2. Real quantum repeaters solve this problem using **entanglement distillation** (also called entanglement purification): taking many noisy Bell pairs and distilling them into fewer, higher-fidelity pairs. In 2–3 sentences, explain conceptually how this would help a repeater chain. You don't need to know the protocol details — just the idea.

3. The lecture mentioned that a practical quantum internet requires entangled pair sources, quantum memories, and entanglement swapping. Which of these three is the main bottleneck today, and why? (2–3 sentences — think about what you know from the hardware lectures.)

**Your answer:**

*(double-click to edit)*

---

**Debrief:** You've built the two key primitives for a quantum repeater: teleportation and entanglement swapping. You've seen that teleportation works regardless of which Bell state is shared (the correction table just changes), and that chaining entanglement swaps distributes entanglement over arbitrary distances — at least in principle. In practice, noise accumulates with each swap, motivating the need for entanglement distillation and quantum error correction. These are active areas of research and the frontier of the quantum internet.

---

**Before submitting:** `Kernel → Restart & Run All` and confirm zero errors. Upload this `.ipynb` to Gradescope.